# SEMANTIC4X3 — nguồn đã khóa, chỉ tiền kiểm (NO SCIENCE)

Hợp đồng V2: bốn nhánh × ba seed, H768 đã niêm phong; L0019 giữ nguyên. Notebook này **không có ô execute** và không tạo CLAIM. T4 Colab Free chỉ được kiểm trong ô cuối. Nếu preflight HOLD, không chạy huấn luyện.

**Trước khi dùng notebook:** chạy `PACK_TTM_SEMANTIC4X3_H768_PRIVATE_V1.ps1` trên Windows. Tải **duy nhất** file ZIP H768 đã kiểm SHA lên `MyDrive/TTM/ttm-small-pilot/TTM_SEMANTIC4X3_H768_PRIVATE_V1_20261110.zip`. ZIP CAUSAL6 cũ đã có sẵn trên Drive, không phải tải lại. Không upload private H768 lên GitHub.


## Ô 1 — Lấy các nguồn tư nhân từ Drive, xác minh SHA và loại H512 cũ


In [ ]:
# CELL 1 — Mount Drive and prepare exact pinned private input + frozen H768.
# No science CLAIM or model training here.
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
from pathlib import Path
import hashlib, zipfile
from google.colab import drive

drive.mount('/content/drive')
PILOT = Path('/content/drive/MyDrive/TTM/ttm-small-pilot')
OLD = PILOT/'TTM_GEMMA2_CAUSAL6_INPUT_PRIVATE_V1_20261109.zip'
NEW = PILOT/'TTM_SEMANTIC4X3_H768_PRIVATE_V1_20261110.zip'
SRC = Path('/content/ttm_semantic4x3_source_v2')
INP = SRC/'input'
CODE = SRC/'code'
H768 = INP/'h768'
RUN = PILOT/'ttm_gemma2_semantic4x3_3seed_2048x4096_h768_20261110_v2'

expected_old_zip_sha='2c6cb72f35eba72d6014f15914d6e9792cbf0bf6774d93e3ff5c53e5c6b9ba9a'
files = {
 'checkpoint.pt':'cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b',
 'candidate_initialization.pt':'c9fc798d8880c7e76636258052e9885dbb086d7b462a5b6bee5cff8b582973c3',
 'T4_PRIVATE_TRAIN_VAL_WINDOWS.npz':'184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913',
 'T4_PRIVATE_PACK_MANIFEST.json':'ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650',
 'src/p02_exact.py':'e0fcac285918b66b13c8f694e6d625a0691c9b59221b43043e63f46b25ddcc07',
 'src/ttm_embeddinggemma2_swap.py':'d11da61e9d7bb040f3263e680f026817a513553c186ab243e184ffaf6e68cd39',
 'src/ttm_directswap2048_gpu_once.py':'0994f0cd91cb504f2db8478f55963813a85469807f2921e38c196248f323759e'
}
h768files={
 'H768_PUBLIC_SEAL.json':'9725b1254cde81b77e23b85ab479a08cafc14af0a7a536c7722ca87c7f2a5aa0',
 'H768_MEMBERSHIP_PRIVATE.json':'0ab671ae59ff4300d8257f2689261d6328afd7563c0aa66b363bc22348c1628f',
 'H768_WINDOWS_PRIVATE.npz':'1eb5b54b4bb7744e23fffd0fd755d5c91853a773dcc0d2219f777b47ba6a53df'
}
def sha(b):return hashlib.sha256(b).hexdigest()
def file_sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for c in iter(lambda:f.read(1<<20),b''):h.update(c)
 return h.hexdigest()

assert OLD.is_file(), 'HOLD_OLD_PRIVATE_ZIP_MISSING'
assert NEW.is_file(), 'HOLD_H768_PRIVATE_ZIP_NOT_UPLOADED'
assert not RUN.exists(), 'HOLD_RUN_DIRECTORY_EXISTS_CLAIM_OR_EVIDENCE'
assert file_sha(OLD)==expected_old_zip_sha, 'HOLD_OLD_ZIP_SHA'
assert not SRC.exists(), 'HOLD_SOURCE_DIR_EXISTS_DO_NOT_OVERWRITE'

def unzip_whitelist(archive,expected,prefix,target):
 with zipfile.ZipFile(archive,'r') as z:
  entries={}
  for i in z.infolist():
   normalized=i.filename.replace(chr(92),'/')
   if i.is_dir(): continue
   if normalized in entries:raise RuntimeError('HOLD_ZIP_DUPLICATE_ENTRY')
   entries[normalized]=i
  expected_keys={prefix+k for k in expected}
  assert expected_keys.issubset(entries.keys()), 'HOLD_ZIP_REQUIRED_ENTRY_MISSING'
  if prefix=='h768/':assert set(entries)==expected_keys,'HOLD_H768_ZIP_UNEXPECTED_ENTRY'
  for name,digest in expected.items():
   data=z.read(entries[prefix+name])
   assert sha(data)==digest, 'HOLD_ZIP_FILE_SHA_'+name
   dest=target/name
   dest.parent.mkdir(parents=True,exist_ok=True)
   with dest.open('xb') as fd:fd.write(data)
   assert file_sha(dest)==digest, 'HOLD_STAGED_SHA_'+name

INP.mkdir(parents=True,exist_ok=False)
unzip_whitelist(OLD,files,'input/',INP)
H768.mkdir(parents=True,exist_ok=False)
unzip_whitelist(NEW,h768files,'h768/',H768)
print('TTM_SEMANTIC4X3_SOURCE_BUNDLE=PASS_SOURCE_SHA7_H768_SHA3')
print('H768=SEALED_768_STORIES_186655_TARGETS_NOT_SCORED')
print('STUDY_CLAIM=NOT_CREATED')

## Ô 2 — Chọn đúng mã công khai từ commit đã ghim, chạy synthetic fixtures (không model training)


In [ ]:
# CELL 2 — Fetch immutable GitHub runner and QA scripts, verify SHA before execution.
import urllib.request, subprocess, sys, hashlib
CODE.mkdir(parents=True,exist_ok=True)
commit='081ea478c7b6fed2748002e0637c041d4fdc21d8'
scripts={
  'ttm_semantic4x3_runner.py':'9a824e453195cafc530e2289d3b8a87e354faf8905ca73b2f29b79144d657a8d',
  'ttm_semantic4x3_independent_readback.py':'51732bd62cfd13bf8956f11ee862b939d0dc78a62c58cd3d655dc05b566764e2',
  'ttm_semantic4x3_balanced_null_cpu_qa.py':'0d71f38da356056be50a18d36f589774190178fd321e6d2e3970413d96238a2d',
}
for name,digest in scripts.items():
  path=CODE/name
  if not path.exists():
    url=('https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/'+commit+
         '/tools/embeddinggemma2_swap/semantic4x3_v2/'+name)
    with urllib.request.urlopen(url,timeout=60) as r: body=r.read()
    assert sha(body)==digest,'HOLD_GITHUB_CODE_SHA_'+name
    with path.open('xb') as f:f.write(body)
  assert file_sha(path)==digest,'HOLD_LOCAL_CODE_SHA_'+name

for name,mode in [('ttm_semantic4x3_runner.py','fixture'),
                  ('ttm_semantic4x3_independent_readback.py','fixture'),
                  ('ttm_semantic4x3_balanced_null_cpu_qa.py','fixture')]:
  r=subprocess.run([sys.executable,'-B',str(CODE/name),'--mode',mode],
                   text=True,capture_output=True,check=False)
  print(name,r.stdout[-500:])
  if r.returncode!=0:
    print(r.stderr[-2500:]);raise RuntimeError('HOLD_FIXTURE_'+name)
print('TTM_SEMANTIC4X3_SYNTHETIC_FIXTURES=PASS_NO_CLAIM')

## Ô 3 — Kiểm tra GPU Tesla T4 và quota Drive thật, sau đó PRE-CLAIM; chưa được phép execute


In [ ]:
# CELL 3 — Integrated real-source, H768 seal, quota and Free T4 preflight.
# Does NOT create STUDY_CLAIM. No score on H768. No model training.
from google.colab import auth
import subprocess, sys, torch

OWNER_VERIFIED_FREE_GIB=9.0 # Conservative owner attestation: approx 10 GB decimal available.
OWNER_CONFIRMS_COLAB_FREE_T4=True
assert OWNER_CONFIRMS_COLAB_FREE_T4,'HOLD_FREE_T4_OWNER_CONFIRMATION'
assert not RUN.exists(),'HOLD_EXISTING_RUN_CLAIM'

# Sign in ONCE here as part of the same preflight, so the runner can read
# Google Drive storageQuota live through about.get without a separate notebook.
auth.authenticate_user()
runner=CODE/'ttm_semantic4x3_runner.py'
args=[sys.executable,'-B',str(runner),'--mode','preflight',
      '--input',str(INP),'--h768',str(H768),'--code-dir',str(CODE),
      '--drive-output',str(RUN),
      '--owner-verified-free-gib',str(OWNER_VERIFIED_FREE_GIB),
      '--free-colab-confirmed']
res=subprocess.run(args,text=True,capture_output=True,check=False)
print('=== STDOUT ===\n'+res.stdout[-18000:])
print('=== STDERR ===\n'+res.stderr[-18000:])
print('PREFLIGHT_EXIT_CODE=',res.returncode)
if res.returncode==0 and not RUN.exists():
 print('TTM_SEMANTIC4X3_PRECLAIM=PASS_NO_SCIENCE_CONSUMED')
else:
 print('TTM_SEMANTIC4X3_PRECLAIM=HOLD_DO_NOT_RUN_SCIENCE')


**Sau khi PASS:** Trình log và trạng thái lên chat để có quyết định science riêng. Không tự thêm cell `--mode execute`, không `Run all` sau khi tạo CLAIM.
